# Sentinel Omega — Jupyter Launcher

Launch the Sentinel Omega orquestador in Jupyter with real-time monitoring.

**Nota**: Sin Telegram. Todos los modelos son ONNX optimizados para GPU/CPU.

## 1. Setup & Imports

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
import logging

# Disable Telegram in Jupyter
os.environ['JUPYTER_ENABLED'] = 'true'
os.environ['JUPYTER_DISABLE_TELEGRAM'] = 'true'

# Add repo to path
repo_path = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(repo_path))

print(f'Repo path: {repo_path}')
print(f'Python path: {sys.path[0]}')

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger('SentinelOmega.Jupyter')

In [ ]:
# Import Sentinel Omega modules
try:
    from sentinel_omega.config.sentinel_config import config
    from sentinel_omega.config.onnx_config import onnx_config
    from sentinel_omega.core.onnx_engine import ONNXModelLoader, ONNXBotInference
    logger.info(f'✓ Sentinel Omega v{config.version}')
    logger.info(f'✓ ONNX Config loaded: {onnx_config.models_dir}')
except ImportError as e:
    logger.error(f'Failed to import Sentinel Omega modules: {e}')
    raise

## 2. Initialize ONNX Models

In [ ]:
from sentinel_omega.config.onnx_config import ONNXRuntimeConfig

# Create ONNX model loader
runtime_cfg = ONNXRuntimeConfig()
loader = ONNXModelLoader(runtime_cfg, onnx_config.models_dir)

# Try to load all enabled models
models_loaded = {}
for bot_name, model_cfg in onnx_config.get_enabled_models().items():
    session = loader.load_model(model_cfg)
    if session:
        models_loaded[bot_name] = session
        logger.info(f'✓ {bot_name} model ready')
    else:
        logger.warning(f'✗ {bot_name} model not found')

logger.info(f'\nLoaded {len(models_loaded)}/{len(onnx_config.get_enabled_models())} models')

## 3. Initialize Database

In [ ]:
from sentinel_omega.infrastructure.database.schema import initialize_database

# Create data directory if needed
data_dir = Path('sentinel_omega/data')
data_dir.mkdir(parents=True, exist_ok=True)

db_path = config.databases.geodynamic_db
logger.info(f'Database: {db_path}')

# Initialize schema
try:
    conn = initialize_database(db_path)
    logger.info('✓ Database initialized')
except Exception as e:
    logger.error(f'Failed to initialize database: {e}')

## 4. Run Single Cycle

In [ ]:
from sentinel_omega.orchestrator import Orchestrator

# Create orchestrator
orchestrator = Orchestrator(
    config=config,
    models=models_loaded,
    disable_telegram=True  # No Telegram in Jupyter
)

logger.info('Starting Orchestrator...')

try:
    result = orchestrator.run_cycle()
    logger.info(f'✓ Cycle completed')
    logger.info(f'Result: {result}')
except Exception as e:
    logger.error(f'Cycle failed: {e}', exc_info=True)

## 5. Run Multiple Cycles (Real-time Monitoring)

In [ ]:
import time
from IPython.display import clear_output

# Configuration
CYCLE_INTERVAL = 300  # seconds (5 minutes)
MAX_CYCLES = 5        # Run 5 cycles

logger.info(f'Starting {MAX_CYCLES} cycles with {CYCLE_INTERVAL}s interval...')

cycle_results = []

for cycle_num in range(MAX_CYCLES):
    try:
        start = datetime.now()
        logger.info(f'\n=== Cycle {cycle_num+1}/{MAX_CYCLES} @ {start.isoformat()} ===')
        
        result = orchestrator.run_cycle()
        
        elapsed = (datetime.now() - start).total_seconds()
        logger.info(f'✓ Completed in {elapsed:.1f}s')
        
        cycle_results.append({
            'cycle': cycle_num + 1,
            'timestamp': start,
            'elapsed': elapsed,
            'result': result
        })
        
        # Wait for next cycle (except after last)
        if cycle_num < MAX_CYCLES - 1:
            logger.info(f'Waiting {CYCLE_INTERVAL}s until next cycle...')
            time.sleep(CYCLE_INTERVAL)
    except Exception as e:
        logger.error(f'Cycle {cycle_num+1} failed: {e}', exc_info=True)

logger.info('\n✓ All cycles completed!')

# Summary
df_results = pd.DataFrame(cycle_results)
print('\n=== Cycle Summary ===')
print(df_results[['cycle', 'elapsed', 'timestamp']])

## 6. Query Results from Database

In [ ]:
from sentinel_omega.infrastructure.database.repository import Repository

# Get latest cycle data
repo = Repository(db_path)

# Get latest detections
detections = repo.get_recent_detections(limit=10)
logger.info(f'\nRecent detections: {len(detections)}')
for det in detections:
    logger.info(f'  - {det}')

# Get latest cycles
cycles = repo.get_recent_cycles(limit=5)
logger.info(f'\nRecent cycles: {len(cycles)}')
for cyc in cycles:
    logger.info(f'  - {cyc}')

## 7. Visualize Data (Optional)

In [ ]:
import matplotlib.pyplot as plt

# Plot cycle times
plt.figure(figsize=(10, 4))
plt.plot(df_results['cycle'], df_results['elapsed'], marker='o', label='Cycle time (s)')
plt.xlabel('Cycle Number')
plt.ylabel('Elapsed Time (seconds)')
plt.title('Sentinel Omega — Cycle Performance')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 8. Shutdown & Cleanup

In [ ]:
# Close database connection
try:
    repo.close()
    logger.info('✓ Database connection closed')
except:
    pass

logger.info('✓ Jupyter session complete')